In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.dataset_window import DownscaleWindowDataset
from torch.utils.data import DataLoader

In [2]:
train_ds = DownscaleWindowDataset(
    "../.data/downscaling_splits/train_norm.nc",
    window_size=3,
    use_elev=True,
    use_date=True
)

print("dataset length:", len(train_ds))

x, y, mask = train_ds[0]
print("x shape   :", x.shape)
print("y shape   :", y.shape)
print("mask shape:", mask.shape)
print("dtype     :", x.dtype, y.dtype, mask.dtype)

dataset length: 1746
x shape   : torch.Size([6, 240, 311])
y shape   : torch.Size([1, 240, 311])
mask shape: torch.Size([1, 240, 311])
dtype     : torch.float32 torch.float32 torch.float32


In [3]:
loader = DataLoader(train_ds, batch_size=2, shuffle=True)
xb, yb, mb = next(iter(loader))

print("xb:", xb.shape)
print("yb:", yb.shape)
print("mb:", mb.shape)

xb: torch.Size([2, 6, 240, 311])
yb: torch.Size([2, 1, 240, 311])
mb: torch.Size([2, 1, 240, 311])


In [4]:
from src.model_unet import UNet

model = UNet(in_channels=6, out_channels=1, base_ch=32)
out = model(xb)

print("xb :", xb.shape)
print("out:", out.shape)
print("yb :", yb.shape)

xb : torch.Size([2, 6, 240, 311])
out: torch.Size([2, 1, 240, 311])
yb : torch.Size([2, 1, 240, 311])


In [5]:
import torch
from src.model_unet import UNet

model = UNet(in_channels=6, out_channels=1, base_ch=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

pred = model(xb)
loss = ((pred - yb) ** 2 * mb).sum() / mb.sum().clamp_min(1.0)

print(loss.item())

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("One U-Net training step passed.")

0.6735687255859375
One U-Net training step passed.
